In [3]:
import os
import pandas as pd
import numpy as np

print("Current working directory:")
print(os.getcwd())

Current working directory:
c:\Users\ASUS\Downloads\3D_Human_Pose_NCKH\phase1_2d_baseline


In [4]:
CSV_DIR = "../data/2_extracted_2d"

csv_files = [f for f in os.listdir(CSV_DIR) if f.endswith(".csv")]

print("Number of CSV files:", len(csv_files))
print("First 5 files:")
print(csv_files[:5])

Number of CSV files: 5483
First 5 files:
['Fall_f_raw_b_1_B_D_0001.csv', 'Fall_f_raw_b_1_B_D_0002.csv', 'Fall_f_raw_b_1_B_D_0003.csv', 'Fall_f_raw_b_1_B_D_0004.csv', 'Fall_f_raw_b_1_B_D_0005.csv']


In [5]:
sample_path = os.path.join(CSV_DIR, csv_files[0])

df_sample = pd.read_csv(sample_path)

print("Sample file:", csv_files[0])
print("Shape:", df_sample.shape)
df_sample.head()

Sample file: Fall_f_raw_b_1_B_D_0001.csv
Shape: (89, 35)


,frame,x0,y0,x1,y1,x2,y2,x3,y3,x4,...,x12,y12,x13,y13,x14,y14,x15,y15,x16,y16
0,0,1115.062866,482.301086,1118.723511,471.453827,1114.744629,467.069550,1102.419067,437.462036,1087.692993,...,854.419006,482.598541,755.448364,632.255737,737.119507,611.638672,464.921570,738.099854,450.203796,742.105347
1,1,1124.277100,484.882599,1124.451050,473.394836,1126.286377,470.247711,1100.747070,437.777985,1100.417480,...,860.444641,480.641846,754.882263,630.390625,743.972900,614.395630,458.776886,754.806152,445.809265,760.247803
2,2,1133.458740,482.752899,1129.986816,471.776825,1133.788452,471.235779,1104.280273,438.365112,1107.900635,...,870.833679,490.181488,774.803650,631.988098,761.317017,625.895264,483.089478,730.985779,472.366241,731.921814
3,3,1138.404541,486.282227,1133.653564,474.073608,1137.891846,475.147797,1108.258057,439.880737,1110.960693,...,876.925415,488.708191,819.824219,625.531982,776.237000,626.797485,530.630005,716.554871,474.819855,724.166321
4,4,1137.641235,487.493164,1132.632935,476.117004,1137.679199,476.289612,1106.719238,444.506012,1111.005737,...,881.202637,497.229858,830.887756,620.341187,771.785889,633.047852,565.203979,713.817810,481.958221,738.396790


In [11]:
data_list = []

for idx, file in enumerate(csv_files, start=1):
    file_path = os.path.join(CSV_DIR, file)
    df = pd.read_csv(file_path)

    # Layer 1 label:
    # 0 = Not_Fall
    # 1 = Fall

    # Layer 2 label:
    # 0 = Sitting
    # 1 = Sleeping
    # 2 = Standing
    # 3 = Walking
    # -1 = Fall, not used for action model

    if file.startswith("Fall"):
        binary_label = 1
        action_label = -1
        action_name = "Fall"

    elif file.startswith("Not_Fall_Sit"):
        binary_label = 0
        action_label = 0
        action_name = "Sitting"

    elif file.startswith("Not_Fall_Sleep"):
        binary_label = 0
        action_label = 1
        action_name = "Sleeping"

    elif file.startswith("Not_Fall_Stand"):
        binary_label = 0
        action_label = 2
        action_name = "Standing"

    elif file.startswith("Not_Fall_Walking"):
        binary_label = 0
        action_label = 3
        action_name = "Walking"

    else:
        print("Skipped unknown file:", file)
        continue

    df["source_file"] = file
    df["binary_label"] = binary_label
    df["action_label"] = action_label
    df["action_name"] = action_name

    data_list.append(df)

    if idx % 500 == 0:
        print(f"Loaded {idx}/{len(csv_files)} files")

master_df = pd.concat(data_list, ignore_index=True)

print("Master dataset shape:", master_df.shape)

print("\nBinary label distribution:")
print(master_df[["source_file", "binary_label"]].drop_duplicates()["binary_label"].value_counts())

print("\nAction label distribution:")
print(
    master_df[master_df["action_label"] != -1][["source_file", "action_label"]]
    .drop_duplicates()["action_label"]
    .value_counts()
    .sort_index()
)

master_df.head()

Loaded 500/5483 files
Loaded 1000/5483 files
Loaded 1500/5483 files
Loaded 2000/5483 files
Loaded 2500/5483 files
Loaded 3000/5483 files
Loaded 3500/5483 files
Loaded 4000/5483 files
Loaded 4500/5483 files
Loaded 5000/5483 files
Master dataset shape: (651989, 39)

Binary label distribution:
binary_label
1    2998
0    2485
Name: count, dtype: int64

Action label distribution:
action_label
0    959
1    523
2    513
3    490
Name: count, dtype: int64


,frame,x0,y0,x1,y1,x2,y2,x3,y3,x4,...,x14,y14,x15,y15,x16,y16,source_file,binary_label,action_label,action_name
0,0,1115.062866,482.301086,1118.723511,471.453827,1114.744629,467.069550,1102.419067,437.462036,1087.692993,...,737.119507,611.638672,464.921570,738.099854,450.203796,742.105347,Fall_f_raw_b_1_B_D_0001.csv,1,-1,Fall
1,1,1124.277100,484.882599,1124.451050,473.394836,1126.286377,470.247711,1100.747070,437.777985,1100.417480,...,743.972900,614.395630,458.776886,754.806152,445.809265,760.247803,Fall_f_raw_b_1_B_D_0001.csv,1,-1,Fall
2,2,1133.458740,482.752899,1129.986816,471.776825,1133.788452,471.235779,1104.280273,438.365112,1107.900635,...,761.317017,625.895264,483.089478,730.985779,472.366241,731.921814,Fall_f_raw_b_1_B_D_0001.csv,1,-1,Fall
3,3,1138.404541,486.282227,1133.653564,474.073608,1137.891846,475.147797,1108.258057,439.880737,1110.960693,...,776.237000,626.797485,530.630005,716.554871,474.819855,724.166321,Fall_f_raw_b_1_B_D_0001.csv,1,-1,Fall
4,4,1137.641235,487.493164,1132.632935,476.117004,1137.679199,476.289612,1106.719238,444.506012,1111.005737,...,771.785889,633.047852,565.203979,713.817810,481.958221,738.396790,Fall_f_raw_b_1_B_D_0001.csv,1,-1,Fall


In [12]:
OUTPUT_PATH = "../data/master_dataset.csv"

master_df.to_csv(OUTPUT_PATH, index=False)

print("Saved master dataset to:", OUTPUT_PATH)

Saved master dataset to: ../data/master_dataset.csv


In [8]:
print(master_df.columns)
print(master_df["label"].value_counts())
print(master_df["label"].value_counts(normalize=True))

Index(['frame', 'x0', 'y0', 'x1', 'y1', 'x2', 'y2', 'x3', 'y3', 'x4', 'y4',
       'x5', 'y5', 'x6', 'y6', 'x7', 'y7', 'x8', 'y8', 'x9', 'y9', 'x10',
       'y10', 'x11', 'y11', 'x12', 'y12', 'x13', 'y13', 'x14', 'y14', 'x15',
       'y15', 'x16', 'y16', 'source_file', 'label'],
      dtype='object')
label
0    382294
1    269695
Name: count, dtype: int64
label
0    0.58635
1    0.41365
Name: proportion, dtype: float64


In [9]:
missing_values = master_df.isnull().sum()

print("Total missing values:", missing_values.sum())
missing_values[missing_values > 0]

Total missing values: 0


Series([], dtype: int64)

In [10]:
video_label_count = master_df[["source_file", "label"]].drop_duplicates()

print(video_label_count["label"].value_counts())
print(video_label_count["label"].value_counts(normalize=True))

label
1    2998
0    2485
Name: count, dtype: int64
label
1    0.546781
0    0.453219
Name: proportion, dtype: float64
